# Testosterone screening thresholds: can the MHANES findings be replicated with public data?

**Source paper.** Hennigar SR, Heymsfield SB, Bukhari AS, Lieberman HR,
Berryman CE. *Testosterone Screening Thresholds in Active-Duty US Male
Service Members.* JAMA, published online 15 Oct 2026,
doi:10.1001/jama.2026.19794. Using the Military Health and Nutrition
Examination Study (MHANES), the authors estimate that 25.7% (95% CI
16.8–34.5) of active-duty Army men would have a single morning total
testosterone below 300 ng/dL and 14.7% (9.7–19.7) below 264 ng/dL.

## Is there a public MHANES data file?

**No.** MHANES (NCT06380322) is a Pennington Biomedical / US Army
Research Institute of Environmental Medicine study, not a CDC survey.
ClinicalTrials.gov records *IPD sharing: No*. The protocol paper
(Berryman et al., Curr Dev Nutr 2026) says deidentified data "may be
made available to external investigators through a controlled-access
process" under DoD data-governance rules, on application. The JAMA
letter's Supplement 2 is a data-sharing statement behind the paywall.
Nothing is downloadable.

## What this project does instead

MHANES was modelled on NHANES, and NHANES 2021–2023 has serum total
testosterone and SHBG (LC-MS/MS), BMI, and a flag for the morning
fasting exam session. Notebook 02 takes NHANES men aged 18–59 drawn in
the morning session, rakes their weights to the 2021 active-duty Army
age and race/ethnicity distribution, and recomputes every quantity the
paper reports: the prevalence table, means, odds ratios per 5 years of
age and 5 BMI units, and the testosterone-by-age scatterplot. The result
is a civilian benchmark for the Army numbers rather than a replication
of them.

## Notebooks

* [01_download_prepare.ipynb](01_download_prepare.ipynb) — pull NHANES
  2021–2023 (and 2011–2016) files, merge, derive variables, save parquet.
* [02_army_calibrated_prevalence.ipynb](02_army_calibrated_prevalence.ipynb)
  — calibration, tables, regression, figures, sensitivity analyses.
* `survey_utils.py` — Taylor-linearised means, raking, survey-weighted
  logistic regression.

## Key results


In [1]:
import json
import pandas as pd
k = json.load(open('outputs/key_results.json'))

rows = []
for key, v in k['prev'].items():
    s, t = key.split('|'); p = k['paper_prev'][key]
    rows.append({'subgroup': s, 'cutoff': f'<{t} ng/dL',
                 'MHANES Army men (paper) % (95% CI)': f"{p['est']:.1f} ({p['lo']:.1f}-{p['hi']:.1f})",
                 'NHANES 2021-23 men, Army-calibrated % (95% CI)': f"{v['est']:.1f} ({v['lo']:.1f}-{v['hi']:.1f})"})
pd.DataFrame(rows).set_index(['subgroup', 'cutoff'])


MHANES Army men (paper) % (95% CI)  \
subgroup         cutoff                                          
Overall          <300 ng/dL                   25.7 (16.8-34.5)   
                 <264 ng/dL                    14.7 (9.7-19.7)   
<30 y            <300 ng/dL                   19.5 (11.1-28.0)   
                 <264 ng/dL                    11.4 (4.6-18.1)   
>=30 y           <300 ng/dL                   32.8 (24.6-40.9)   
                 <264 ng/dL                   18.6 (15.3-21.9)   
Nonobese 19-39 y <264 ng/dL                     9.7 (4.6-14.8)   

                            NHANES 2021-23 men, Army-calibrated % (95% CI)  
subgroup         cutoff                                                     
Overall          <300 ng/dL                                12.7 (7.4-18.0)  
                 <264 ng/dL                                 6.5 (2.8-10.3)  
<30 y            <300 ng/dL                                10.7 (5.2-16.2)  
                 <264 ng/dL                                  4.6 (0.1-9.0)  
>=30 y           <300 ng/dL                                15.6 (8.3-23.0)  
                 <264 ng/dL                                 9.3 (4.6-14.1)  
Nonobese 19-39 y <264 ng/dL                                  0.6 (0.0-1.3)

In [2]:
pd.DataFrame(k['or_table']).set_index(['cutoff', 'term'])


NHANES OR, single predictor NHANES OR, age + BMI jointly  \
cutoff term                                                             
<300   age/5            1.19 (1.07-1.34)             1.15 (0.99-1.35)   
       bmi/5            1.71 (1.25-2.35)             1.70 (1.26-2.29)   
<264   age/5            1.20 (1.03-1.41)             1.16 (0.95-1.42)   
       bmi/5            1.88 (1.32-2.68)             1.88 (1.35-2.62)   

             MHANES OR (paper)  
cutoff term                     
<300   age/5  1.29 (1.17-1.43)  
       bmi/5  1.90 (1.28-2.83)  
<264   age/5  1.20 (1.14-1.27)  
       bmi/5  1.82 (1.07-3.10)

In [3]:
pd.DataFrame(k['means']).T


,nhanes,paper
"mean age, y",29.2 (28.2-30.1),29.9 (SE 0.6)
mean BMI,28.1 (26.9-29.3),28.4 (SE 0.2)
"total testosterone, ng/dL",549 (500-597),405 (368-442)
"SHBG, nmol/L",30.6 (28.7-32.5),—
"SHBG, µg/mL (×0.095)",2.91 (2.73-3.09),3.24 (2.73-3.75)
"free testosterone, ng/dL",12.55 (10.88-14.23),8.74 (7.70-9.81)
"median total testosterone, ng/dL",476 (unweighted),—


In [4]:
pd.DataFrame(k['sensitivity']).set_index('sample')


,n,<300 % (95% CI),<264 % (95% CI),mean TT ng/dL (95% CI)
sample,,,,
"2021-23 morning, Army-calibrated (primary)",784,12.7 (7.4-18.0),6.5 (2.8-10.3),549 (500-597)
"2021-23 morning, US civilian weights",784,16.4 (11.0-21.7),9.8 (5.6-14.1),512 (479-546)
"2021-23 all sessions (morning + afternoon/evening), Army-calibrated",1443,17.5 (13.4-21.5),11.0 (7.6-14.3),505 (466-545)
"2021-23 afternoon/evening only, Army-calibrated",659,22.8 (17.4-28.2),15.7 (11.2-20.2),457 (420-494)
"2011-16 pooled morning, 8 testosterone-Rx users excluded, Army-calibrated",2557,14.2 (11.3-17.0),9.0 (6.8-11.1),487 (472-503)
"2011-16 pooled morning, US civilian weights",2557,17.8 (15.2-20.4),11.4 (9.4-13.5),458 (447-469)


Prevalence if every NHANES value is scaled down by a constant factor,
to gauge how much of the gap an assay difference could explain:


In [5]:
pd.DataFrame(k['assay_shift']).set_index('scale factor')


,<300 % (95% CI),<264 % (95% CI),>916 % (95% CI)
scale factor,,,
1.00,12.7 (7.4-18.0),6.5 (2.8-10.3),6.6 (2.4-10.8)
0.90,17.8 (12.3-23.2),11.6 (6.7-16.5),3.2 (0.9-5.4)
0.85,20.1 (14.8-25.5),13.6 (8.2-19.1),2.3 (0.4-4.3)
0.80,23.1 (18.0-28.3),16.2 (10.4-22.1),1.8 (0.3-3.2)
0.74 (equal means),29.0 (21.8-36.2),20.3 (14.9-25.7),1.0 (0.0-2.2)
MHANES (paper),25.7 (16.8-34.5),14.7 (9.7-19.7),1.3 (0.0-3.2)


## Reading the comparison

* The civilian prevalence of a single low morning value, standardised
  to Army age and race/ethnicity, is about **half** the MHANES estimate at
  both cutoffs (12.7% vs 25.7% below 300 ng/dL). The confidence
  intervals barely overlap.
* The whole NHANES distribution sits higher, not just the low tail:
  mean total testosterone 549 vs 405 ng/dL, and 6.6% vs 1.3% of men
  above 916 ng/dL.
* The **BMI gradient replicates** (OR per 5 BMI units 1.7–1.9 in both
  datasets). The age gradient is weaker in NHANES (1.19 vs 1.29 per 5 y
  for the 300 cutoff).
* The most likely reason for the gap is the **assay**: NHANES uses
  LC-MS/MS, MHANES used an IMMULITE 2000 immunoassay that reads lower in
  the male range. Scaling the NHANES values down by 15–20% closes most
  of the difference, and a 20–26% scaling brackets the paper's values. Occupational stressors on soldiers are
  the other candidate. The two cannot be separated without the MHANES
  data.
* Draw timing matters as much as the cutoff: afternoon/evening NHANES
  draws raise the prevalence of a value below 300 ng/dL from 12.7% to
  22.8%.

The paper's policy conclusion, that routine screening will flag many men
who do not have hypogonadism, is supported either way: even the civilian
benchmark puts one man in eight below 300 ng/dL on a single morning draw.

## Reproduce

```
uv venv && uv pip install -r requirements.txt
.venv/bin/jupyter nbconvert --to notebook --execute --inplace 01_download_prepare.ipynb 02_army_calibrated_prevalence.ipynb 00_overview.ipynb
```
